# Notebook 5 — Comprehensive Evaluation & Scalability Analysis
## HMDA 2023 Loan Denial Prediction

**Objective:** Perform bootstrap confidence interval analysis, strong/weak scaling experiments,
residual/error analysis, and business-oriented evaluation metrics for all trained models.

---

In [15]:
# ============================================================
# Imports & SparkSession
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.classification import (
    LogisticRegressionModel, DecisionTreeClassificationModel,
    RandomForestClassificationModel, GBTClassificationModel,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json, time, os, warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .appName("HMDA_2023_Evaluation_5")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

Spark 3.5.5 | Cores: 2


26/03/02 02:26:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/02 02:26:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/02 02:26:48 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [16]:
# ============================================================
# Load Data & Models
# ============================================================

DATA_DIR = "../data/processed"
MODEL_DIR = f"{DATA_DIR}/models"
RESULTS_DIR = f"{DATA_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

test_df = spark.read.parquet(f"{DATA_DIR}/test_features.parquet").cache()
test_count = test_df.count()
feature_dim = test_df.first()['features'].size

# Load previous results
with open(f"{DATA_DIR}/model_results_4.json") as f:
    ALL_RESULTS = json.load(f)

with open(f"{DATA_DIR}/feature_metadata.json") as f:
    feature_meta = json.load(f)

# Add weight column for models that need it
label_counts = {row['label']: row['count']
                for row in test_df.groupBy('label').count().collect()}
n_orig = label_counts[0.0]
n_denied = label_counts[1.0]
total = n_orig + n_denied
w0 = total / (2.0 * n_orig)
w1 = total / (2.0 * n_denied)

test_w = test_df.withColumn(
    'weight', F.when(F.col('label') == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()

print(f"Test set: {test_count:,} rows x {feature_dim} features")
print(f"Class weights: originated={w0:.4f}, denied={w1:.4f}")
print(f"Models available: {list(ALL_RESULTS.keys())}")

Test set: 1,533,558 rows x 145 features
Class weights: originated=0.6749, denied=1.9293
Models available: ['M0_Baseline', 'M1_NaiveBayes', 'M2_LogisticRegression', 'M3_LinearSVM', 'M4_DecisionTree']


## Section 1 — Resource Allocation Justification

| Parameter | Value | Justification |
|:----------|:------|:--------------|
| `master` | `local[2]` | Limits concurrent tasks to 2 cores to prevent OOM on 8GB machine with 4GB dataset |
| `spark.driver.memory` | `6g` | Accommodates ~4GB dataset in memory with headroom for caching, shuffle buffers, and JVM overhead |
| `spark.executor.memory` | `4g` | In local mode, executors share the driver JVM; 4GB ensures in-memory operations without disk spill |
| `spark.driver.maxResultSize` | `2g` | Allows large collect()/toPandas() for evaluation metrics and confusion matrices |
| `spark.sql.shuffle.partitions` | `200` | Default value handles ~7.7M records well; AQE dynamically coalesces empty partitions |
| `KryoSerializer` | enabled | 2-10x faster and more compact than Java serialization, reducing shuffle I/O |
| `Arrow` | enabled | Enables zero-copy transfer between Spark and Pandas for toPandas() calls |
| `AQE` | enabled | Runtime optimization for partition coalescing and skew handling |
| `TVS parallelism` | 1 | Sequential model evaluation prevents OOM from multiple concurrent model copies |

In [17]:
# ============================================================
# Print Resource Allocation
# ============================================================

print("=" * 70)
print("RESOURCE ALLOCATION")
print("=" * 70)
configs = [
    ("spark.master", spark.conf.get("spark.master")),
    ("spark.driver.memory", spark.conf.get("spark.driver.memory", "N/A")),
    ("spark.driver.maxResultSize", spark.conf.get("spark.driver.maxResultSize", "N/A")),
    ("spark.sql.shuffle.partitions", spark.conf.get("spark.sql.shuffle.partitions", "200")),
    ("spark.serializer", spark.conf.get("spark.serializer", "default")),
    ("spark.sql.adaptive.enabled", spark.conf.get("spark.sql.adaptive.enabled", "false")),
]
for name, val in configs:
    print(f"  {name:<45} = {val}")

RESOURCE ALLOCATION
  spark.master                                  = local[2]
  spark.driver.memory                           = 6g
  spark.driver.maxResultSize                    = 2g
  spark.sql.shuffle.partitions                  = 200
  spark.serializer                              = org.apache.spark.serializer.KryoSerializer
  spark.sql.adaptive.enabled                    = true


## Storage Format Justification

| Format | Compression | Read Speed | Column Pruning | Predicate Pushdown | Chosen? |
|:-------|:-----------|:-----------|:--------------|:------------------|:--------|
| CSV | None (~4 GB) | Slow (full scan) | No | No | Raw ingestion only |
| JSON | Minimal | Slow (parse overhead) | No | No | No |
| ORC | Good (~700 MB) | Fast | Yes | Yes | Viable alternative |
| Avro | Moderate (~1.2 GB) | Fast (row-oriented) | No | Limited | No |
| **Parquet + Snappy** | **~800 MB (80% reduction)** | **Fast (~250 MB/s decompression)** | **Yes** | **Yes** | **Yes** |

**Why Parquet?** The HMDA dataset has 99 columns but most queries access 10-20. Parquet's columnar format enables column pruning (read only needed columns) and predicate pushdown (filter at storage level). With state_code partitioning, geographic queries read only relevant partitions. Snappy compression provides fast decompression for iterative ML training.

## Section 2 — Bootstrap Confidence Intervals

In [18]:
# ============================================================
# Bootstrap Confidence Intervals for PR-AUC
# ============================================================

N_BOOTSTRAP = 50
SAMPLE_FRACTION = 0.5
SEED_BASE = 42

eval_pr = BinaryClassificationEvaluator(
    labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderPR'
)
eval_roc = BinaryClassificationEvaluator(
    labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

# Load best models (use M-prefixed names consistent with notebook 4)
models_to_evaluate = {}
try:
    models_to_evaluate['M2_LogisticRegression'] = LogisticRegressionModel.load(f"{MODEL_DIR}/best_lr")
except: print("LR model not found")
try:
    models_to_evaluate['M4_DecisionTree'] = DecisionTreeClassificationModel.load(f"{MODEL_DIR}/best_dt")
except: print("DT model not found")

print(f"Loaded {len(models_to_evaluate)} models for bootstrap analysis")

bootstrap_results = {}

for model_name, model in models_to_evaluate.items():
    print(f"\nBootstrapping {model_name} ({N_BOOTSTRAP} iterations)...")
    pr_scores = []
    roc_scores = []
    
    preds = model.transform(test_w)
    
    for i in range(N_BOOTSTRAP):
        sample = preds.sample(withReplacement=True, fraction=SAMPLE_FRACTION, seed=SEED_BASE + i)
        pr_auc = eval_pr.evaluate(sample)
        roc_auc = eval_roc.evaluate(sample)
        pr_scores.append(pr_auc)
        roc_scores.append(roc_auc)
        if (i + 1) % 10 == 0:
            print(f"  Iteration {i+1}/{N_BOOTSTRAP}")
    
    bootstrap_results[model_name] = {
        'pr_auc_mean': float(np.mean(pr_scores)),
        'pr_auc_std': float(np.std(pr_scores)),
        'pr_auc_ci_lower': float(np.percentile(pr_scores, 2.5)),
        'pr_auc_ci_upper': float(np.percentile(pr_scores, 97.5)),
        'roc_auc_mean': float(np.mean(roc_scores)),
        'roc_auc_ci_lower': float(np.percentile(roc_scores, 2.5)),
        'roc_auc_ci_upper': float(np.percentile(roc_scores, 97.5)),
        'pr_scores': [float(s) for s in pr_scores],
        'roc_scores': [float(s) for s in roc_scores],
    }
    preds.unpersist()

print("\n" + "=" * 80)
print("BOOTSTRAP CONFIDENCE INTERVALS (95%)")
print("=" * 80)
print(f"{'Model':<30} {'PR-AUC Mean':>12} {'CI Lower':>10} {'CI Upper':>10} {'ROC-AUC Mean':>13}")
print("-" * 80)
for name, r in sorted(bootstrap_results.items(), key=lambda x: -x[1]['pr_auc_mean']):
    print(f"  {name:<28} {r['pr_auc_mean']:>12.4f} {r['pr_auc_ci_lower']:>10.4f} {r['pr_auc_ci_upper']:>10.4f} {r['roc_auc_mean']:>13.4f}")

Loaded 2 models for bootstrap analysis

Bootstrapping M2_LogisticRegression (50 iterations)...


  Iteration 10/50


  Iteration 20/50


  Iteration 30/50


  Iteration 40/50


  Iteration 50/50

Bootstrapping M4_DecisionTree (50 iterations)...
  Iteration 10/50
  Iteration 20/50
  Iteration 30/50
  Iteration 40/50
  Iteration 50/50

BOOTSTRAP CONFIDENCE INTERVALS (95%)
Model                           PR-AUC Mean   CI Lower   CI Upper  ROC-AUC Mean
--------------------------------------------------------------------------------
  M2_LogisticRegression              0.9988     0.9988     0.9989        0.9996
  M4_DecisionTree                    0.9982     0.9981     0.9984        0.9994


In [19]:
# ============================================================
# Pairwise Significance Testing
# ============================================================

print("=" * 70)
print("PAIRWISE SIGNIFICANCE (Non-overlapping CIs)")
print("=" * 70)

model_names = sorted(bootstrap_results.keys())
pairwise = []
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        a = model_names[i]
        b = model_names[j]
        a_upper = bootstrap_results[a]['pr_auc_ci_upper']
        a_lower = bootstrap_results[a]['pr_auc_ci_lower']
        b_upper = bootstrap_results[b]['pr_auc_ci_upper']
        b_lower = bootstrap_results[b]['pr_auc_ci_lower']
        
        overlaps = not (a_lower > b_upper or b_lower > a_upper)
        result = 'NOT significant' if overlaps else 'SIGNIFICANT'
        pairwise.append({'pair': f"{a} vs {b}", 'significant': not overlaps})
        print(f"  {a} vs {b}: {result}")

sig_count = sum(1 for p in pairwise if p['significant'])
print(f"\n{sig_count}/{len(pairwise)} pairs show statistically significant differences.")

PAIRWISE SIGNIFICANCE (Non-overlapping CIs)
  M2_LogisticRegression vs M4_DecisionTree: SIGNIFICANT

1/1 pairs show statistically significant differences.


In [20]:
# ============================================================
# Bootstrap CI Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

names = sorted(bootstrap_results.keys(), key=lambda x: bootstrap_results[x]['pr_auc_mean'], reverse=True)
means = [bootstrap_results[n]['pr_auc_mean'] for n in names]
lowers = [bootstrap_results[n]['pr_auc_mean'] - bootstrap_results[n]['pr_auc_ci_lower'] for n in names]
uppers = [bootstrap_results[n]['pr_auc_ci_upper'] - bootstrap_results[n]['pr_auc_mean'] for n in names]

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
y_pos = range(len(names))

ax.barh(y_pos, means, xerr=[lowers, uppers], color=colors[:len(names)],
        capsize=5, edgecolor='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('PR-AUC')
ax.set_title('Model Comparison: PR-AUC with 95% Bootstrap Confidence Intervals')
ax.set_xlim(0.99, 1.001)

for i, (m, name) in enumerate(zip(means, names)):
    ax.text(m + 0.0001, i, f'{m:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: bootstrap_ci.png')

Saved: bootstrap_ci.png


## Section 3 — Bottleneck Identification

| Pipeline Stage | Dominant Bottleneck | Evidence | Mitigation |
|:--------------|:-------------------|:---------|:-----------|
| CSV Ingestion (NB1) | **I/O** | 4 GB CSV read from disk; 12.6s load time | Convert to Parquet once; subsequent reads are 3-5x faster |
| Schema Validation (NB1) | **Computation** | Type inference on 99 columns × 11.5M rows | Use explicit schema instead of inferSchema; validate in single pass |
| EDA Statistical Tests (NB2) | **Computation** | Chi-square and t-tests across 75+ categorical columns | Sample-based approximations; cache DataFrame before repeated queries |
| Feature Engineering (NB3) | **Shuffle** | StringIndexer + OneHotEncoder pipeline across 25 categoricals; 29 pipeline stages | Pipeline fit is a single pass; use AQE for partition coalescing |
| Train/Test Split (NB3) | **Shuffle** | Stratified sampling uses subtract() which triggers full shuffle | Use sampleBy() for stratification; 192s for 7.7M rows |
| Model Training (NB4) | **Computation + Memory** | TVS with 12 hyperparameter combos × 7.7M rows; OOM at 8 cores | Reduce to local[2], parallelism=1; sequential model evaluation |
| GBT Training (NB4) | **Computation** | 3,928s for GBT with iterative boosting (20-50 iterations) | Limit maxIter; use early stopping via validation metrics |
| RandomForest Training (NB4) | **Memory** | 100 trees × deep splits require parallel tree storage | Limit numTrees; unpersist TVS after each model |
| Evaluation (NB5) | **I/O + Computation** | 50 bootstrap iterations × model.transform() on test set | Cache predictions before bootstrap sampling |

## Section 4 — Scaling Analysis

In [21]:
# ============================================================
# Strong Scaling: Fixed data, varying partitions
# ============================================================

print("=" * 70)
print("STRONG SCALING EXPERIMENT")
print("=" * 70)

# Use a 10% sample for scaling experiments
scale_sample = test_w.sample(fraction=0.1, seed=42).cache()
scale_count = scale_sample.count()
print(f"Sample size: {scale_count:,} rows")

# Pick a model for scaling (LR is fastest to transform)
scale_model = None
for name in ['M2_LogisticRegression', 'M4_DecisionTree']:
    if name in models_to_evaluate:
        scale_model = models_to_evaluate[name]
        scale_model_name = name
        break

strong_scaling = []
partition_counts = [4, 8, 16, 32, 64]

if scale_model:
    for n_parts in partition_counts:
        repartitioned = scale_sample.repartition(n_parts)
        start = time.time()
        preds = scale_model.transform(repartitioned)
        _ = preds.count()  # Force evaluation
        elapsed = time.time() - start
        strong_scaling.append({'partitions': n_parts, 'time_s': round(elapsed, 3)})
        print(f"  Partitions={n_parts:>3} | Time={elapsed:.3f}s")
        preds.unpersist()
else:
    print("No model available for scaling experiments")

print(f"\nOptimal: {min(strong_scaling, key=lambda x: x['time_s'])} (if available)")

STRONG SCALING EXPERIMENT
Sample size: 153,017 rows
  Partitions=  4 | Time=0.118s
  Partitions=  8 | Time=0.114s
  Partitions= 16 | Time=0.125s
  Partitions= 32 | Time=0.195s
  Partitions= 64 | Time=0.198s

Optimal: {'partitions': 8, 'time_s': 0.114} (if available)


In [22]:
# ============================================================
# Weak Scaling: Proportional data increase
# ============================================================

print("=" * 70)
print("WEAK SCALING EXPERIMENT")
print("=" * 70)

weak_scaling = []
fractions = [0.25, 0.50, 0.75, 1.0]

if scale_model:
    for frac in fractions:
        subset = scale_sample.sample(fraction=frac, seed=42)
        subset_count = subset.count()
        start = time.time()
        preds = scale_model.transform(subset)
        _ = preds.count()
        elapsed = time.time() - start
        weak_scaling.append({
            'fraction': f"{int(frac*100)}%",
            'records': subset_count,
            'time_s': round(elapsed, 3)
        })
        print(f"  {int(frac*100):>3}% | {subset_count:>10,} rows | Time={elapsed:.3f}s")
        preds.unpersist()

WEAK SCALING EXPERIMENT
   25% |     38,344 rows | Time=0.070s
   50% |     76,725 rows | Time=0.074s
   75% |    115,044 rows | Time=0.069s
  100% |    153,017 rows | Time=0.069s


In [23]:
# ============================================================
# Scaling Visualization
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Strong scaling
if strong_scaling:
    parts = [s['partitions'] for s in strong_scaling]
    times = [s['time_s'] for s in strong_scaling]
    ax1.plot(parts, times, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Partitions')
    ax1.set_ylabel('Time (seconds)')
    ax1.set_title('Strong Scaling (Fixed Data, Varying Partitions)')
    ax1.grid(True, alpha=0.3)

# Weak scaling
if weak_scaling:
    records = [s['records'] for s in weak_scaling]
    times = [s['time_s'] for s in weak_scaling]
    ax2.plot(records, times, 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Records')
    ax2.set_ylabel('Time (seconds)')
    ax2.set_title('Weak Scaling (Proportional Data Increase)')
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/scaling_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: scaling_analysis.png')

Saved: scaling_analysis.png


## Section 5 — Training Cost Estimation

In [24]:
# ============================================================
# Training Cost Estimation
# ============================================================

COST_PER_HOUR = 0.50  # Estimated cloud compute cost per hour

print("=" * 70)
print("TRAINING COST ESTIMATION")
print("=" * 70)
print(f"{'Model':<30} {'Training Time':>15} {'Est. Cost':>12}")
print("-" * 60)

cost_data = []
for name, metrics in sorted(ALL_RESULTS.items(), key=lambda x: x[1].get('Train_Time_s', 0)):
    t = metrics.get('Train_Time_s', 0)
    cost = (t / 3600) * COST_PER_HOUR
    cost_data.append({'model': name, 'time_s': t, 'cost': cost})
    print(f"  {name:<28} {t:>14.1f}s ${cost:>10.4f}")

total_cost = sum(c['cost'] for c in cost_data)
total_time = sum(c['time_s'] for c in cost_data)
print(f"\n  {'TOTAL':<28} {total_time:>14.1f}s ${total_cost:>10.4f}")

TRAINING COST ESTIMATION
Model                            Training Time    Est. Cost
------------------------------------------------------------
  M0_Baseline                             0.0s $    0.0000
  M1_NaiveBayes                          40.5s $    0.0056
  M3_LinearSVM                          188.1s $    0.0261
  M4_DecisionTree                       437.1s $    0.0607
  M2_LogisticRegression                 642.5s $    0.0892

  TOTAL                                1308.2s $    0.1817


## Section 6 — Error Analysis by Subgroup

In [25]:
# ============================================================
# Confusion Matrix Analysis for Best Model
# ============================================================

# Find best model by PR-AUC
best_model_name = max(
    [(k, v) for k, v in ALL_RESULTS.items() if k != 'M0_Baseline'],
    key=lambda x: x[1].get('PR-AUC', 0)
)[0]

best_metrics = ALL_RESULTS[best_model_name]
cm = best_metrics.get('Confusion', {})

print("=" * 70)
print(f"BEST MODEL: {best_model_name}")
print("=" * 70)
print(f"\nConfusion Matrix:")
print(f"                  Predicted")
print(f"                  Originated  Denied")
print(f"  Actual Orig.    {cm.get('TN', 0):>10,}  {cm.get('FP', 0):>8,}")
print(f"  Actual Denied   {cm.get('FN', 0):>10,}  {cm.get('TP', 0):>8,}")

tp = cm.get('TP', 0)
fp = cm.get('FP', 0)
fn = cm.get('FN', 0)
tn = cm.get('TN', 0)
total = tp + fp + fn + tn

print(f"\nBusiness Impact Metrics:")
print(f"  True Positive Rate (Denial Recall):  {tp/(tp+fn)*100:.1f}% — correctly identified denials")
print(f"  False Positive Rate:                 {fp/(fp+tn)*100:.2f}% — originated loans incorrectly flagged")
print(f"  False Negative Rate:                 {fn/(fn+tp)*100:.2f}% — denied loans missed")
print(f"  Precision (Denial):                  {tp/(tp+fp)*100:.1f}% — of predicted denials, actually denied")
print(f"  Overall Accuracy:                    {(tp+tn)/total*100:.2f}%")

BEST MODEL: M2_LogisticRegression

Confusion Matrix:
                  Predicted
                  Originated  Denied
  Actual Orig.     1,131,189     4,927
  Actual Denied        4,762   392,680

Business Impact Metrics:
  True Positive Rate (Denial Recall):  98.8% — correctly identified denials
  False Positive Rate:                 0.43% — originated loans incorrectly flagged
  False Negative Rate:                 1.20% — denied loans missed
  Precision (Denial):                  98.8% — of predicted denials, actually denied
  Overall Accuracy:                    99.37%


In [26]:
# ============================================================
# Model Comparison Visualization
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Filter out baseline
model_data = {k: v for k, v in ALL_RESULTS.items() if k != 'M0_Baseline'}
names = list(model_data.keys())

# PR-AUC comparison
pr_aucs = [model_data[n].get('PR-AUC', 0) for n in names]
axes[0].barh(names, pr_aucs, color='steelblue')
axes[0].set_xlabel('PR-AUC')
axes[0].set_title('PR-AUC by Model')
axes[0].set_xlim(0.98, 1.0)

# Denial F1 comparison
denial_f1s = [model_data[n].get('Denial_F1', 0) for n in names]
axes[1].barh(names, denial_f1s, color='forestgreen')
axes[1].set_xlabel('Denial F1')
axes[1].set_title('Denial F1 by Model')
axes[1].set_xlim(0.97, 1.0)

# Training time comparison
times = [model_data[n].get('Train_Time_s', 0) for n in names]
axes[2].barh(names, times, color='coral')
axes[2].set_xlabel('Training Time (s)')
axes[2].set_title('Training Time by Model')

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

Saved: model_comparison.png


In [27]:
# ============================================================
# Save All Evaluation Results
# ============================================================

evaluation_report = {
    'best_model': best_model_name,
    'evaluation_metrics': ALL_RESULTS,
    'bootstrap_confidence_intervals': bootstrap_results,
    'pairwise_significance': pairwise,
    'strong_scaling': strong_scaling,
    'weak_scaling': weak_scaling,
    'training_costs': cost_data,
    'feature_dimensions': feature_dim,
    'test_set_size': test_count,
}

report_path = f"{RESULTS_DIR}/final_evaluation_report.json"
with open(report_path, 'w') as f:
    json.dump(evaluation_report, f, indent=2, default=str)

# print(f"Evaluation report saved: {report_path}")
print(f"  Models evaluated: {len(ALL_RESULTS)}")
print(f"  Bootstrap iterations: {N_BOOTSTRAP}")
print(f"  Strong scaling points: {len(strong_scaling)}")
print(f"  Weak scaling points: {len(weak_scaling)}")

  Models evaluated: 5
  Bootstrap iterations: 50
  Strong scaling points: 5
  Weak scaling points: 4


In [28]:
# ============================================================
# Stop Spark
# ============================================================

test_w.unpersist()
test_df.unpersist()
scale_sample.unpersist()

spark.stop()
print("Spark stopped. Notebook 5 complete.")

Spark stopped. Notebook 5 complete.
